# Airbnb in New York City: Sharing Economy or Professional Rental Market?

This notebook explores whether Airbnb in New York City still behaves like a sharing-economy platform, or whether a substantial part of the platform is dominated by more professional hosts.

The core idea is simple: Airbnb started as a way for people to occasionally rent out space in their homes. But in a city like New York, listings may also be managed more like a business. Since the dataset does not contain a direct label for "professional host", this notebook uses observable behavioral proxies such as:

- the number of listings controlled by the same host,
- the type of listing (entire home vs private room),
- the yearly availability of the listing,
- and the spatial concentration of listings across boroughs and neighborhoods.

The goal is not to prove legal business status, but to understand whether Airbnb usage in NYC looks more like casual sharing, more like continuous commercial activity, or a mix of both.

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import HeatMap

## 1. Dataset and limitations

The data comes from Inside Airbnb and contains detailed information on Airbnb listings in New York City.

Important limitations:

- The dataset does **not** directly identify whether a host is a professional business.
- The `price` column is unusable in our copy of the dataset, so this analysis does not rely on pricing.
- Some indicators can only be interpreted as **proxies** of professionalization. For example, a host with multiple listings may still not be a company, but such behavior is less consistent with the original home-sharing idea.
- This is listing-level data, not booking-level data, so availability is a useful signal but not a direct measure of actual rentals.

In [8]:
df = pd.read_csv("../data/processed/cleaned_listings.csv")

## 2. Basic dataset overview

Before constructing any indicators, we first inspect the size and scope of the dataset.

In [9]:
print("Number of listings:", len(df))
print("Number of unique hosts:", df["host_id"].nunique())
print("Room types:", df["room_type"].unique())
print("Boroughs:", df["neighbourhood_group_cleansed"].unique())

Number of listings: 36418
Number of unique hosts: 21448
Room types: ['Entire home/apt' 'Private room' 'Hotel room' 'Shared room']
Boroughs: ['Manhattan' 'Brooklyn' 'Bronx' 'Queens' 'Staten Island']


The dataset contains listing-level observations, including:

- host identifier,
- room type,
- borough and neighborhood,
- geographic coordinates,
- yearly availability,
- minimum number of nights,
- and number of reviews.

These variables are enough to study whether different parts of the platform look more casual or more professionalized.

## 3. Defining host categories

A central idea in this notebook is that hosts with only one listing are more consistent with the original "sharing economy" model, while hosts controlling several listings may reflect more professionalized activity.

We therefore create host categories based on the total number of listings controlled by each host:

- **1 listing**
- **2–4 listings**
- **5+ listings**

This is not a perfect definition, but it is a reasonable and transparent proxy.

In [10]:
host_counts = df.groupby("host_id").size().rename("host_listing_count")
df = df.merge(host_counts, on="host_id")

In [11]:
def host_category(n):
    if n == 1:
        return "1 listing"
    elif n <= 4:
        return "2–4 listings"
    else:
        return "5+ listings"

df["host_category"] = df["host_listing_count"].apply(host_category)

In [12]:
df[["host_id", "host_listing_count", "host_category"]].head()

,host_id,host_listing_count,host_category
0,2845,3,2–4 listings
1,15991,1,1 listing
2,16104,2,2–4 listings
3,16800,1,1 listing
4,17571,2,2–4 listings


## 4. Host concentration

We begin by asking a simple question:

**How concentrated is the platform among hosts?**

If almost everyone has exactly one listing, the platform would look closer to casual sharing. If many listings belong to hosts with several properties, the platform starts to resemble a professional rental market.

In [13]:
host_dist = df[["host_id", "host_listing_count"]].drop_duplicates()
host_dist["host_listing_count"].describe()

count    21448.000000
mean         1.697967
std         10.772713
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max       1210.000000
Name: host_listing_count, dtype: float64

In [14]:
print("Share of hosts with 1 listing:", (host_dist["host_listing_count"] == 1).mean())
print("Share of hosts with >1 listing:", (host_dist["host_listing_count"] > 1).mean())
print("Share of hosts with 5+ listings:", (host_dist["host_listing_count"] >= 5).mean())

Share of hosts with 1 listing: 0.8362551286833271
Share of hosts with >1 listing: 0.1637448713166729
Share of hosts with 5+ listings: 0.029559865721745616


In [15]:
fig = px.histogram(
    host_dist[host_dist["host_listing_count"] <= 10],
    x="host_listing_count",
    nbins=10,
    title="Number of Listings per Host (Zoom on 1–10)",
    labels={"host_listing_count": "Number of listings controlled by host", "count": "Number of hosts"}
)

fig.update_layout(
    xaxis=dict(dtick=1),
    template="plotly_white"
)

fig.show()

### Interpretation

The distribution is extremely right-skewed. Most hosts only manage one listing, which is consistent with casual hosting. However, the long tail matters: a smaller group of hosts controls several listings, which suggests that part of Airbnb in NYC behaves more like a professionalized rental market than a simple sharing platform.